In [52]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

In [53]:
os.makedirs("outputs", exist_ok=True)
sns.set(style="whitegrid")

In [54]:
df = pd.read_csv('titanic.csv')
print("Shape of dataset:",df.shape)
print(df.head())

Shape of dataset: (891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450 

In [55]:
print("\n---Info---")
print(df.info())


---Info---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [56]:
print("\n--- Describe ---")
print(df.describe(include="all"))


--- Describe ---
        PassengerId    Survived      Pclass                     Name   Sex  \
count    891.000000  891.000000  891.000000                      891   891   
unique          NaN         NaN         NaN                      891     2   
top             NaN         NaN         NaN  Braund, Mr. Owen Harris  male   
freq            NaN         NaN         NaN                        1   577   
mean     446.000000    0.383838    2.308642                      NaN   NaN   
std      257.353842    0.486592    0.836071                      NaN   NaN   
min        1.000000    0.000000    1.000000                      NaN   NaN   
25%      223.500000    0.000000    2.000000                      NaN   NaN   
50%      446.000000    0.000000    3.000000                      NaN   NaN   
75%      668.500000    1.000000    3.000000                      NaN   NaN   
max      891.000000    1.000000    3.000000                      NaN   NaN   

               Age       SibSp       Parch  T

In [57]:
print("\n--- Missing values per column ---")
missing = df.isnull().sum()
print(missing[missing > 0])


--- Missing values per column ---
Age         177
Cabin       687
Embarked      2
dtype: int64


In [58]:
# Visualize missingness

In [59]:
plt.figure(figsize=(10,6))
sns.heatmap(df.isnull(), cbar=False,cmap = "viridis")
plt.title("Missing Value Heatmap(Before Cleaning)")
plt.tight_layout()
plt.savefig("outputs/missing_heatmap_before.png")
plt.close()

In [60]:
plt.figure(figsize=(8, 5))
missing[missing > 0].sort_values(ascending=False).plot(kind="bar", color="salmon")
plt.title("Count of Missing Values by Column")
plt.ylabel("Number of Missing Values")
plt.tight_layout()
plt.savefig("outputs/missing_counts_bar.png")
plt.close()

In [61]:
# Handle Missing Values

In [62]:
df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

In [63]:
df["Age"] = df["Age"].fillna(df["Age"].median())

In [64]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [65]:
df["Has_Cabin"] = df["Cabin"].notnull().astype(int)
df.drop(columns=["Cabin"], inplace=True)

In [66]:
print("\n--- Missing values after cleaning ---")
print(df.isnull().sum())


--- Missing values after cleaning ---
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
Has_Cabin      0
dtype: int64


In [67]:
# Feature engineering

In [68]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

In [69]:
# Demographic & survival analysis

In [72]:
plt.figure(figsize=(6,4))
sns.barplot(data=df, x="Sex",y="Survived")
plt.title("Survival Rate by Sex")
plt.tight_layout()
plt.savefig("Outputs/survival_by_sex.png")
plt.close()

In [73]:
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="Pclass", y="Survived")
plt.title("Survival Rate by Passenger Class")
plt.tight_layout()
plt.savefig("outputs/survival_by_pclass.png")
plt.close()

In [74]:
plt.figure(figsize=(7,5))
sns.histplot(df["Age"], bins=30,kde=True,color="teal")
plt.title("Age Distribution")
plt.tight_layout()
plt.savefig("outputs/age_distribution_after.png")
plt.close()

In [75]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Pclass", y="Fare")
plt.title("Fare Distribution by Passenger Class")
plt.tight_layout()
plt.savefig("outputs/fare_by_class.png")
plt.close()

In [76]:
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("outputs/correlation_heatmap.png")
plt.close()

In [78]:
print("Mean Age by Survival")
print(df.groupby("Survived")["Age"].mean())

Mean Age by Survival
Survived
0    29.737705
1    28.108684
Name: Age, dtype: float64


In [79]:
print("Mean Fare by Class")
print(df.groupby("Pclass")["Fare"].mean())

Mean Fare by Class
Pclass
1    84.154687
2    20.662183
3    13.675550
Name: Fare, dtype: float64


In [80]:
print("Survival Rate by Sex & Class")
print(df.groupby(["Sex", "Pclass"])["Survived"].mean())

Survival Rate by Sex & Class
Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64


In [82]:
df.to_csv("outputs/titanic_cleaned.csv", index=False)
print("Cleaned dataset saved to outputs/titanic_cleaned.csv")
print("All charts saved inside the 'outputs' folder.")

Cleaned dataset saved to outputs/titanic_cleaned.csv
All charts saved inside the 'outputs' folder.
